# Notebook 0: Preparação Completa de Dados Cinematográficos

**Objetivo:** Ler, integrar e limpar dados de sessões SCB/ANCINE (2023-2025)

**Output:** `df_sessoes_limpo.parquet`

---

## Estrutura
1. Configuração e Imports
2. Leitura de Dados (Sessões, Salas, Tipos, IBGE)
3. Integração (Joins com chave composta)
4. Regras de Negócio
5. Validações e Exportação

## 1. Configuração e Imports

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import requests
import zipfile
import io
import fnmatch
from pathlib import Path
from datetime import datetime
import json
import warnings

warnings.filterwarnings('ignore')

print("✓ Bibliotecas importadas")
print(f"  Pandas: {pd.__version__}")
print(f"  Polars: {pl.__version__}")

✓ Bibliotecas importadas
  Pandas: 2.3.3
  Polars: 1.35.1


## 2. Definição de Parâmetros

In [2]:
# Anos a processar
ANOS = [2023, 2024, 2025]

# Períodos de cada ano cinematográfico
PERIODOS = {
    2023: ('2023-01-05', '2024-01-03'),
    2024: ('2024-01-04', '2025-01-01'),
    2025: ('2025-01-02', '2025-09-30')
}

# Caminhos
CAMINHO_TIPOS_SESSAO = '../Bases/AG_SCB_SESSAO_REDUZIDO.parquet'
CAMINHO_OUTPUT = '../Bases/df_sessoes_limpo.parquet'
CAMINHO_DICIONARIO = '../Bases/dicionario_df_sessoes_limpo.json'

# URLs ANCINE
URL_BILHETERIA_BASE = "https://dados.ancine.gov.br/dados-abertos/bilheteria-diaria-obras-por-exibidoras-csv.zip"
URL_SALAS = "https://dados.ancine.gov.br/dados-abertos/salas-de-exibicao-e-complexos.csv"

print("✓ Parâmetros configurados")
print(f"  Anos: {ANOS}")
for ano, (inicio, fim) in PERIODOS.items():
    print(f"  {ano}: {inicio} até {fim}")

✓ Parâmetros configurados
  Anos: [2023, 2024, 2025]
  2023: 2023-01-05 até 2024-01-03
  2024: 2024-01-04 até 2025-01-01
  2025: 2025-01-02 até 2025-09-30


## 3. Leitura de Dados

### 3.1. Sessões (Bilheteria) - ANCINE

In [3]:
# Schema para leitura dos CSVs de bilheteria
schema_bilheteria = {
    "DATA_EXIBICAO": pl.Date,
    "SESSAO": pl.Datetime,
    "TITULO_ORIGINAL": pl.Utf8,
    "TITULO_BRASIL": pl.Utf8,
    "CPB_ROE": pl.Utf8,
    "AUDIO": pl.Utf8,
    "LEGENDADA": pl.Utf8,
    "PAIS_OBRA": pl.Utf8,
    "REGISTRO_SALA": pl.Utf8,
    "NOME_SALA": pl.Utf8,
    "PUBLICO": pl.Int64,
    "REGISTRO_GRUPO_EXIBIDOR": pl.Utf8,
    "REGISTRO_EXIBIDOR": pl.Utf8,
    "REGISTRO_COMPLEXO": pl.Utf8,
    "MUNICIPIO_SALA_COMPLEXO": pl.Utf8,
    "UF_SALA_COMPLEXO": pl.Utf8,
    "RAZAO_SOCIAL_EXIBIDORA": pl.Utf8,
    "CNPJ_EXIBIDORA": pl.Utf8,
}

print("✓ Schema definido")

✓ Schema definido


In [4]:
def ler_bilheteria_ano(ano):
    """
    Lê dados de bilheteria de um ano específico da ANCINE.
    
    Args:
        ano: Ano a ser lido (2023, 2024, 2025)
    
    Returns:
        DataFrame pandas com dados do ano
    """
    print(f"\n📥 Baixando dados de {ano}...")
    
    # URL do arquivo ZIP
    url = f"https://dados.ancine.gov.br/dados-abertos/bilheteria-diaria-obras-por-exibidoras-csv.zip"
    
    # Download do ZIP
    resposta = requests.get(url)
    resposta.raise_for_status()
    arquivo_zip = zipfile.ZipFile(io.BytesIO(resposta.content))
    
    # Padrão de nomes dos CSVs do ano
    padrao = f"bilheteria-diaria-obras-por-exibidoras-{ano}-*.csv"
    arquivos_csv = sorted([
        nome for nome in arquivo_zip.namelist() 
        if fnmatch.fnmatch(nome, padrao)
    ])
    
    print(f"  Encontrados {len(arquivos_csv)} arquivos CSV")
    
    # Lista para armazenar DataFrames
    lista_df = []
    
    # Ler cada CSV com Polars
    for i, nome_arquivo in enumerate(arquivos_csv):
        print(f"  Lendo: {nome_arquivo}")
        with arquivo_zip.open(nome_arquivo) as arq_csv:
            df_polars = pl.read_csv(
                arq_csv,
                separator=";",
                schema=schema_bilheteria,
                try_parse_dates=True,
                truncate_ragged_lines=True
            )
        
        # Remover cabeçalho duplicado (exceto primeiro arquivo)
        if i == 0:
            lista_df.append(df_polars)
        else:
            lista_df.append(df_polars.slice(1))
    
    # Concatenar todos os DataFrames
    df_final = pl.concat(lista_df)
    
    # Converter para Pandas
    df_pandas = df_final.to_pandas()
    
    # Remover colunas desnecessárias
    df_pandas.drop(columns=['CNPJ_EXIBIDORA', 'NOME_SALA'], inplace=True, errors='ignore')
    
    print(f"  ✓ {len(df_pandas):,} sessões lidas para {ano}")
    
    return df_pandas

print("✓ Função ler_bilheteria_ano() definida")

✓ Função ler_bilheteria_ano() definida


In [5]:
# Ler dados de todos os anos
print("="*70)
print("LEITURA DE SESSÕES (BILHETERIA)")
print("="*70)

lista_sessoes = []

for ano in ANOS:
    df_ano = ler_bilheteria_ano(ano)
    
    # Adicionar coluna de ano cinematográfico
    df_ano['ANO_CINEMA'] = ano
    
    lista_sessoes.append(df_ano)

# Concatenar todos os anos
df_sessoes = pd.concat(lista_sessoes, ignore_index=True)

print(f"\n✅ Total de sessões carregadas: {len(df_sessoes):,}")
print(f"   Período: {df_sessoes['DATA_EXIBICAO'].min()} até {df_sessoes['DATA_EXIBICAO'].max()}")

LEITURA DE SESSÕES (BILHETERIA)

📥 Baixando dados de 2023...
  Encontrados 59 arquivos CSV
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-01-s01.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-01-s02.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-01-s03.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-01-s04.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-01-s05.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-02-s01.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-02-s02.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-02-s03.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-02-s04.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-03-s01.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-03-s02.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-03-s03.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-03-s04.csv
  Lendo: bilheteria-diaria-obras-por-exibidoras-2023-03-s05.csv
  Lendo: bilh

### 3.2. Filtrar por Período de Cada Ano Cinematográfico

In [6]:
print("🗓️ Filtrando por períodos cinematográficos...\n")

# Criar máscara de filtro
mascara = pd.Series(False, index=df_sessoes.index)

for ano, (inicio, fim) in PERIODOS.items():
    data_inicio = pd.to_datetime(inicio)
    data_fim = pd.to_datetime(fim)
    
    # Adicionar à máscara as sessões deste ano
    mascara_ano = (
        (df_sessoes['ANO_CINEMA'] == ano) &
        (df_sessoes['DATA_EXIBICAO'] >= data_inicio) &
        (df_sessoes['DATA_EXIBICAO'] <= data_fim)
    )
    
    n_sessoes = mascara_ano.sum()
    print(f"  {ano}: {n_sessoes:,} sessões ({inicio} até {fim})")
    
    mascara |= mascara_ano

# Aplicar filtro
registros_antes = len(df_sessoes)
df_sessoes = df_sessoes[mascara].copy()
registros_removidos = registros_antes - len(df_sessoes)

print(f"\n✓ Sessões mantidas: {len(df_sessoes):,}")
print(f"✗ Sessões removidas (fora do período): {registros_removidos:,}")

🗓️ Filtrando por períodos cinematográficos...

  2023: 3,990,269 sessões (2023-01-05 até 2024-01-03)
  2024: 4,301,443 sessões (2024-01-04 até 2025-01-01)
  2025: 3,280,640 sessões (2025-01-02 até 2025-09-30)

✓ Sessões mantidas: 11,572,352
✗ Sessões removidas (fora do período): 621,642


### 3.3. Salas - ANCINE

In [7]:
print("="*70)
print("LEITURA DE SALAS")
print("="*70)

# Colunas a manter
colunas_manter = [
    "NOME_SALA",
    "REGISTRO_SALA",
    "CNPJ_SALA",
    "SITUACAO_SALA",
    "DATA_SITUACAO_SALA",
    "DATA_INICIO_FUNCIONAMENTO_SALA",
    "ASSENTOS_SALA",
    "NOME_COMPLEXO",
    "REGISTRO_COMPLEXO",
    "SITUACAO_COMPLEXO",
    "DATA_SITUACAO_COMPLEXO",
    "MUNICIPIO_COMPLEXO",
    "UF_COMPLEXO",
    "COMPLEXO_ITINERANTE",
    "OPERACAO_USUAL",
    "NOME_EXIBIDOR",
    "REGISTRO_EXIBIDOR",
    "CNPJ_EXIBIDOR",
    "SITUACAO_EXIBIDOR",
    "NOME_GRUPO_EXIBIDOR"
]

# Tipos de colunas
tipos_colunas = {
    "NOME_SALA": str,
    "REGISTRO_SALA": str,
    "CNPJ_SALA": str,
    "SITUACAO_SALA": str,
    "ASSENTOS_SALA": "Int64",
    "NOME_COMPLEXO": str,
    "REGISTRO_COMPLEXO": str,
    "SITUACAO_COMPLEXO": str,
    "MUNICIPIO_COMPLEXO": str,
    "UF_COMPLEXO": str,
    "COMPLEXO_ITINERANTE": str,
    "OPERACAO_USUAL": str,
    "NOME_EXIBIDOR": str,
    "REGISTRO_EXIBIDOR": str,
    "CNPJ_EXIBIDOR": str,
    "SITUACAO_EXIBIDOR": str,
    "NOME_GRUPO_EXIBIDOR": str
}

print("\n📥 Baixando dados de salas...")

df_salas = pd.read_csv(
    URL_SALAS,
    sep=";",
    usecols=colunas_manter,
    dtype=tipos_colunas,
    parse_dates=["DATA_SITUACAO_SALA", "DATA_INICIO_FUNCIONAMENTO_SALA", "DATA_SITUACAO_COMPLEXO"],
    dayfirst=True
)

print(f"✓ {len(df_salas):,} salas lidas")
print(f"  Colunas: {len(df_salas.columns)}")

LEITURA DE SALAS

📥 Baixando dados de salas...
✓ 6,303 salas lidas
  Colunas: 20


### 3.4. Tipos de Sessão (Arquivo Local)

In [8]:
print("="*70)
print("LEITURA DE TIPOS DE SESSÃO")
print("="*70)

print(f"\n📂 Lendo arquivo: {CAMINHO_TIPOS_SESSAO}")

df_tipos = pd.read_parquet(CAMINHO_TIPOS_SESSAO)

print(f"✓ {len(df_tipos):,} registros lidos")
print(f"  Colunas: {df_tipos.columns.tolist()}")
print(f"\n📊 Distribuição de tipos:")
print(df_tipos['TIPO_SESSAO'].value_counts())

LEITURA DE TIPOS DE SESSÃO

📂 Lendo arquivo: ../Bases/AG_SCB_SESSAO_REDUZIDO.parquet
✓ 11,115,446 registros lidos
  Colunas: ['ID_SESSAO_CINEMATOGRAFICA', 'DATA_HORA_SESSAO', 'TIPO_SESSAO', 'REGISTRO_SALA', 'CPB_ROE']

📊 Distribuição de tipos:
TIPO_SESSAO
Sessão Regular        11037641
Sessão Privada           37714
Pré-Estreia              37680
Mostra ou Festival        2411
Name: count, dtype: int64


### 3.5. Dados IBGE (API)

Vamos buscar:
- População municipal (Censo 2022)
- PIB municipal (2021 - mais recente disponível)

In [9]:
def buscar_dados_ibge_municipios():
    """
    Busca PIB (2021) e População Censo 2022 de todos os municípios via API IBGE.
    Calcula PIB per capita = PIB Total / População.
    """
    print("🚀 Buscando dados IBGE...")
    
    # 1. Obter lista de municípios
    print("   1. Obtendo lista de municípios...")
    try:
        resp = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/municipios", timeout=30)
        resp.raise_for_status()
        municipios = resp.json()
        ids_municipios = [str(m['id']) for m in municipios]
        print(f"      ✓ {len(ids_municipios)} municípios encontrados")
    except Exception as e:
        print(f"      ❌ Erro: {e}")
        return pd.DataFrame()

    # 2. Configuração - LOTE REDUZIDO PARA 50
    TAMANHO_LOTE = 50  # ← MUDANÇA AQUI (era 100)
    total_lotes = (len(ids_municipios) + TAMANHO_LOTE - 1) // TAMANHO_LOTE
    session = requests.Session()
    dados_consolidados = {}
    
    print(f"   2. Processando {total_lotes} lotes (2 requisições por lote)...")
    
    for i in range(0, len(ids_municipios), TAMANHO_LOTE):
        lote = ids_municipios[i:i + TAMANHO_LOTE]
        str_ids = ",".join(lote)
        localidades = f"N6[{str_ids}]"
        
        # Inicializar dados do lote
        for cod in lote:
            if cod not in dados_consolidados:
                dados_consolidados[cod] = {
                    'codigo_municipio': cod,
                    'nome_municipio': None,
                    'pib_total': None,
                    'populacao': None
                }
        
        try:
            # A. PIB TOTAL (Agregado 5938, Variável 37, Ano 2021)
            r = session.get(
                f"https://servicodados.ibge.gov.br/api/v3/agregados/5938/periodos/2021/variaveis/37?localidades={localidades}",
                timeout=20  # ← AUMENTADO (era 15)
            )
            if r.ok:
                data = r.json()
                if data and 'resultados' in data[0]:
                    for item in data[0]['resultados'][0]['series']:
                        cod = item['localidade']['id']
                        valor = item['serie'].get('2021')
                        dados_consolidados[cod]['pib_total'] = float(valor) if valor else None
                        dados_consolidados[cod]['nome_municipio'] = item['localidade']['nome']
            
            # B. POPULAÇÃO CENSO 2022 (Agregado 9514, Variável 93)
            r = session.get(
                f"https://servicodados.ibge.gov.br/api/v3/agregados/9514/periodos/2022/variaveis/93?localidades={localidades}",
                timeout=20  # ← AUMENTADO (era 15)
            )
            if r.ok:
                data = r.json()
                if data and 'resultados' in data[0]:
                    for item in data[0]['resultados'][0]['series']:
                        cod = item['localidade']['id']
                        valor = item['serie'].get('2022')
                        dados_consolidados[cod]['populacao'] = int(valor) if valor else None
            
            print(".", end="", flush=True)
            
        except Exception as e:
            print("X", end="", flush=True)
            # Em caso de erro, tentar reprocessar este lote com timeout maior
            try:
                print(f"\n   ⚠️ Timeout no lote, tentando novamente...", end="")
                r = session.get(
                    f"https://servicodados.ibge.gov.br/api/v3/agregados/9514/periodos/2022/variaveis/93?localidades={localidades}",
                    timeout=30
                )
                if r.ok:
                    data = r.json()
                    if data and 'resultados' in data[0]:
                        for item in data[0]['resultados'][0]['series']:
                            cod = item['localidade']['id']
                            valor = item['serie'].get('2022')
                            dados_consolidados[cod]['populacao'] = int(valor) if valor else None
                print(" OK", end="")
            except:
                pass
    
    print("\n   3. Convertendo e calculando PIB per capita...")
    
    # Criar DataFrame
    df = pd.DataFrame(list(dados_consolidados.values()))
    
    if df.empty:
        print("      ❌ DataFrame vazio")
        return df
    
    # Calcular PIB per capita
    df['pib_per_capita'] = (df['pib_total'] * 1000) / df['populacao']
    
    # Ordenar colunas
    df = df[['codigo_municipio', 'nome_municipio', 'populacao', 'pib_total', 'pib_per_capita']]
    
    # Estatísticas
    com_pib = df['pib_total'].notna().sum()
    com_pop = df['populacao'].notna().sum()
    com_pib_pc = df['pib_per_capita'].notna().sum()
    
    print(f"✅ {len(df)} municípios processados")
    print(f"   • Com PIB Total (2021): {com_pib}")
    print(f"   • Com População (Censo 2022): {com_pop}")
    print(f"   • Com PIB per capita: {com_pib_pc}")
    
    return df

In [10]:
print("="*70)
print("LEITURA DE DADOS IBGE")
print("="*70)

# Buscar dados consolidados
df_ibge = buscar_dados_ibge_municipios()  # ← SEM o ano="2021"

if not df_ibge.empty:
    print(f"\n✓ Dados IBGE carregados: {len(df_ibge):,} municípios")
else:
    print("\n⚠️ Não foi possível obter dados IBGE")

LEITURA DE DADOS IBGE
🚀 Buscando dados IBGE...
   1. Obtendo lista de municípios...
      ✓ 5571 municípios encontrados
   2. Processando 112 lotes (2 requisições por lote)...
.......................................................................................................X
   ⚠️ Timeout no lote, tentando novamente...........
   3. Convertendo e calculando PIB per capita...
✅ 5571 municípios processados
   • Com PIB Total (2021): 5570
   • Com População (Censo 2022): 5570
   • Com PIB per capita: 5570

✓ Dados IBGE carregados: 5,571 municípios


In [11]:
print("="*70)
print("ANÁLISE DE MUNICÍPIOS SEM DADOS IBGE")
print("="*70)

# Municípios sem população
sem_pop = df_ibge[df_ibge['populacao'].isna()].copy()
print(f"\n📊 {len(sem_pop)} municípios sem dados de população:")
if not sem_pop.empty:
    print(sem_pop[['codigo_municipio', 'nome_municipio']].to_string())
else:
    print("   (Nenhum município sem dados)")

# Verificar se esses municípios têm salas de cinema
if not sem_pop.empty:
    print("\n🎬 Verificando se têm salas de cinema...")
    
    # Limpar nomes dos municípios sem população (remover (UF) do final)
    sem_pop['nome_limpo'] = sem_pop['nome_municipio'].fillna('').str.replace(r'\s*\([A-Z]{2}\)\s*$', '', regex=True).str.upper().str.strip()
    
    # Verificar no df_salas
    if 'MUNICIPIO_COMPLEXO' in df_salas.columns:
        municipios_com_sala = df_salas['MUNICIPIO_COMPLEXO'].str.upper().str.strip().unique()
        
        tem_sala = []
        for idx, row in sem_pop.iterrows():
            nome = row['nome_limpo']
            if nome and nome in municipios_com_sala:
                tem_sala.append(row['nome_municipio'])
        
        if tem_sala:
            print(f"\n⚠️ ATENÇÃO: {len(tem_sala)} municípios sem dados TÊM salas de cinema:")
            for nome in tem_sala:
                print(f"   • {nome}")
            print("\n   Recomendo reprocessar esses municípios!")
        else:
            print(f"\n✓ ÓTIMO! Nenhum município sem dados tem sala de cinema")
            print(f"✓ Pode prosseguir tranquilamente!")
    else:
        print("\n⚠️ df_salas ainda não está disponível, não foi possível verificar")
    
    print(f"\n📋 Lista completa dos {len(sem_pop)} municípios sem população:")
    for idx, row in sem_pop.iterrows():
        print(f"   {row['codigo_municipio']}: {row['nome_municipio']}")
else:
    print("\n✅ Todos os municípios têm dados de população!")

ANÁLISE DE MUNICÍPIOS SEM DADOS IBGE

📊 1 municípios sem dados de população:
     codigo_municipio nome_municipio
5199          5101837           None

🎬 Verificando se têm salas de cinema...

✓ ÓTIMO! Nenhum município sem dados tem sala de cinema
✓ Pode prosseguir tranquilamente!

📋 Lista completa dos 1 municípios sem população:
   5101837: None


## 4. Integração de Dados

### 4.1. Join: Sessões + Tipos de Sessão

**Importante:** Join com chave composta (CPB_ROE + DATA_HORA_SESSAO + REGISTRO_SALA)

In [12]:
print("="*70)
print("PREPARAÇÃO PARA JOIN")
print("="*70)

print("\n🔄 Renomeando e padronizando tipos...")

# 1. Renomear SESSAO para DATA_HORA_SESSAO
if 'SESSAO' in df_sessoes.columns:
    df_sessoes.rename(columns={'SESSAO': 'DATA_HORA_SESSAO'}, inplace=True)
    print("  ✓ SESSAO → DATA_HORA_SESSAO")

# 2. Carregar e preparar df_tipos
df_tipos = pd.read_parquet(CAMINHO_TIPOS_SESSAO)
print(f"  ✓ Parquet carregado: {len(df_tipos):,} registros")

# 3. Converter timezone
df_tipos['DATA_HORA_SESSAO'] = pd.to_datetime(df_tipos['DATA_HORA_SESSAO'], utc=True)
df_tipos['DATA_HORA_SESSAO'] = df_tipos['DATA_HORA_SESSAO'].dt.tz_convert('America/Sao_Paulo').dt.tz_localize(None)
print("  ✓ Timezone convertido (UTC → Brasília)")

# 4. Padronizar tipos de dados
df_sessoes['DATA_HORA_SESSAO'] = pd.to_datetime(df_sessoes['DATA_HORA_SESSAO']).dt.tz_localize(None)
df_sessoes['REGISTRO_SALA'] = df_sessoes['REGISTRO_SALA'].astype(str)
df_sessoes['CPB_ROE'] = df_sessoes['CPB_ROE'].astype(str)

df_tipos['REGISTRO_SALA'] = df_tipos['REGISTRO_SALA'].astype(str)
df_tipos['CPB_ROE'] = df_tipos['CPB_ROE'].astype(str)

print("\n✓ Tipos padronizados:")
print(f"   df_sessoes['DATA_HORA_SESSAO']: {df_sessoes['DATA_HORA_SESSAO'].dtype}")
print(f"   df_tipos['DATA_HORA_SESSAO']: {df_tipos['DATA_HORA_SESSAO'].dtype}")

PREPARAÇÃO PARA JOIN

🔄 Renomeando e padronizando tipos...
  ✓ SESSAO → DATA_HORA_SESSAO
  ✓ Parquet carregado: 11,115,446 registros
  ✓ Timezone convertido (UTC → Brasília)

✓ Tipos padronizados:
   df_sessoes['DATA_HORA_SESSAO']: datetime64[us]
   df_tipos['DATA_HORA_SESSAO']: datetime64[ns]


In [13]:
print("\n" + "="*70)
print("ALINHANDO PERÍODOS")
print("="*70)

data_min_parquet = df_tipos['DATA_HORA_SESSAO'].min()
data_max_parquet = df_tipos['DATA_HORA_SESSAO'].max()

print(f"\n📅 Período do parquet:")
print(f"  {data_min_parquet} até {data_max_parquet}")

print(f"\n📅 Período de df_sessoes (antes):")
print(f"  {df_sessoes['DATA_HORA_SESSAO'].min()} até {df_sessoes['DATA_HORA_SESSAO'].max()}")

# Filtrar df_sessoes
registros_antes = len(df_sessoes)
df_sessoes = df_sessoes[
    (df_sessoes['DATA_HORA_SESSAO'] >= data_min_parquet) &
    (df_sessoes['DATA_HORA_SESSAO'] <= data_max_parquet)
].copy()

print(f"\n✂️ Sessões removidas (fora do período): {registros_antes - len(df_sessoes):,}")
print(f"✓ Sessões restantes: {len(df_sessoes):,}")


ALINHANDO PERÍODOS

📅 Período do parquet:
  2023-01-05 04:20:00 até 2025-08-22 22:10:00

📅 Período de df_sessoes (antes):
  2023-01-05 04:20:00 até 2025-09-30 23:51:00

✂️ Sessões removidas (fora do período): 464,903
✓ Sessões restantes: 11,107,449


In [14]:
print("\n" + "="*70)
print("JOIN: SESSÕES + TIPOS DE SESSÃO")
print("="*70)

print(f"\nAntes do join:")
print(f"  Sessões: {len(df_sessoes):,}")
print(f"  Tipos: {len(df_tipos):,}")

# Join
df_sessoes = df_sessoes.merge(
    df_tipos[['CPB_ROE', 'DATA_HORA_SESSAO', 'REGISTRO_SALA', 'TIPO_SESSAO']],
    on=['CPB_ROE', 'DATA_HORA_SESSAO', 'REGISTRO_SALA'],
    how='left'
)

print(f"\nApós o join:")
print(f"  Sessões: {len(df_sessoes):,}")

# Cobertura
com_tipo = df_sessoes['TIPO_SESSAO'].notna().sum()
sem_tipo = df_sessoes['TIPO_SESSAO'].isna().sum()

print(f"\n📊 Cobertura:")
print(f"  Com tipo: {com_tipo:,} ({com_tipo/len(df_sessoes)*100:.1f}%)")
print(f"  Sem tipo: {sem_tipo:,} ({sem_tipo/len(df_sessoes)*100:.1f}%)")

if com_tipo > 0:
    print(f"\n📊 Distribuição:")
    print(df_sessoes['TIPO_SESSAO'].value_counts())


JOIN: SESSÕES + TIPOS DE SESSÃO

Antes do join:
  Sessões: 11,107,449
  Tipos: 11,115,446

Após o join:
  Sessões: 11,107,449

📊 Cobertura:
  Com tipo: 11,074,831 (99.7%)
  Sem tipo: 32,618 (0.3%)

📊 Distribuição:
TIPO_SESSAO
Sessão Regular        10997867
Sessão Privada           37579
Pré-Estreia              36980
Mostra ou Festival        2405
Name: count, dtype: int64


In [15]:
print("="*70)
print("IDENTIFICANDO SESSÕES FALTANTES NO PARQUET")
print("="*70)

# RECARREGAR o parquet
print("\n📂 Recarregando parquet...")
df_parquet_check = pd.read_parquet(CAMINHO_TIPOS_SESSAO)
df_parquet_check['DATA_HORA_SESSAO'] = pd.to_datetime(df_parquet_check['DATA_HORA_SESSAO'], utc=True)
df_parquet_check['DATA_HORA_SESSAO'] = df_parquet_check['DATA_HORA_SESSAO'].dt.tz_convert('America/Sao_Paulo').dt.tz_localize(None)
df_parquet_check['REGISTRO_SALA'] = df_parquet_check['REGISTRO_SALA'].astype(str)
df_parquet_check['CPB_ROE'] = df_parquet_check['CPB_ROE'].astype(str)

print(f"✓ Parquet recarregado: {len(df_parquet_check):,} registros")

# Criar chaves compostas
print("\n🔑 Criando chaves compostas...")

df_sessoes['chave_completa'] = (
    df_sessoes['CPB_ROE'].astype(str) + '|' + 
    df_sessoes['REGISTRO_SALA'].astype(str) + '|' + 
    df_sessoes['DATA_HORA_SESSAO'].astype(str)
)

df_parquet_check['chave_completa'] = (
    df_parquet_check['CPB_ROE'].astype(str) + '|' + 
    df_parquet_check['REGISTRO_SALA'].astype(str) + '|' + 
    df_parquet_check['DATA_HORA_SESSAO'].astype(str)
)

# Identificar faltantes
chaves_sessoes = set(df_sessoes['chave_completa'])
chaves_parquet = set(df_parquet_check['chave_completa'])
chaves_faltantes = chaves_sessoes - chaves_parquet

print(f"✓ Análise concluída:")
print(f"  Em df_sessoes: {len(chaves_sessoes):,}")
print(f"  Em parquet: {len(chaves_parquet):,}")
print(f"  Faltantes: {len(chaves_faltantes):,} ({len(chaves_faltantes)/len(chaves_sessoes)*100:.1f}%)")

# Filtrar sessões faltantes
df_faltantes = df_sessoes[df_sessoes['chave_completa'].isin(chaves_faltantes)].copy()

print(f"\n📊 Distribuição das sessões faltantes:")
print(f"\n  Por ano:")
print(df_faltantes['ANO_CINEMA'].value_counts().sort_index())

print(f"\n  Por mês (primeiros 10):")
df_faltantes['MES'] = pd.to_datetime(df_faltantes['DATA_HORA_SESSAO']).dt.to_period('M')
print(df_faltantes['MES'].value_counts().sort_index().head(10))

print(f"\n📋 Exemplos:")
print(df_faltantes[['DATA_HORA_SESSAO', 'CPB_ROE', 'REGISTRO_SALA', 'TITULO_BRASIL']].head(5))



IDENTIFICANDO SESSÕES FALTANTES NO PARQUET

📂 Recarregando parquet...
✓ Parquet recarregado: 11,115,446 registros

🔑 Criando chaves compostas...
✓ Análise concluída:
  Em df_sessoes: 11,107,449
  Em parquet: 11,115,446
  Faltantes: 32,618 (0.3%)

📊 Distribuição das sessões faltantes:

  Por ano:
ANO_CINEMA
2023     3398
2024     4611
2025    24609
Name: count, dtype: int64

  Por mês (primeiros 10):
MES
2023-01    137
2023-02    162
2023-03    142
2023-04    131
2023-05    187
2023-06    148
2023-07    387
2023-08    451
2023-09    446
2023-10    482
Freq: M, Name: count, dtype: int64

📋 Exemplos:
        DATA_HORA_SESSAO         CPB_ROE REGISTRO_SALA  \
3498 2023-01-05 16:00:00  E2200431200000       5002529   
4148 2023-01-05 16:30:00  E2200403600000       5005624   
6922 2023-01-05 19:00:00  E2200403600000       5005624   
7842 2023-01-05 20:00:00  E2200431200000       5002529   
9923 2023-01-05 21:30:00  E2200413300000       5005624   

                         TITULO_BRASIL  
3498 

In [16]:
print("="*70)
print("INVESTIGAÇÃO DOS 32.618 FALTANTES")
print("="*70)

# Pegar os faltantes
df_faltantes_inv = df_sessoes[df_sessoes['TIPO_SESSAO'].isna()].copy()

print(f"\n📊 Total de faltantes: {len(df_faltantes_inv):,}")

# 1. Verificar se essas sessões REALMENTE existem no parquet
print("\n🔍 Teste 1: Verificar existência no parquet")

# Pegar 10 exemplos e buscar no parquet original
df_tipos_check = pd.read_parquet(CAMINHO_TIPOS_SESSAO)
df_tipos_check['DATA_HORA_SESSAO'] = pd.to_datetime(df_tipos_check['DATA_HORA_SESSAO'], utc=True)
df_tipos_check['DATA_HORA_SESSAO'] = df_tipos_check['DATA_HORA_SESSAO'].dt.tz_convert('America/Sao_Paulo').dt.tz_localize(None)
df_tipos_check['REGISTRO_SALA'] = df_tipos_check['REGISTRO_SALA'].astype(str)
df_tipos_check['CPB_ROE'] = df_tipos_check['CPB_ROE'].astype(str)

print("\nTestando 10 sessões faltantes:")
for idx in df_faltantes_inv.head(10).index:
    row = df_sessoes.loc[idx]
    cpb = row['CPB_ROE']
    sala = row['REGISTRO_SALA']
    data = row['DATA_HORA_SESSAO']
    
    # Buscar no parquet
    match = df_tipos_check[
        (df_tipos_check['CPB_ROE'] == cpb) &
        (df_tipos_check['REGISTRO_SALA'] == sala) &
        (df_tipos_check['DATA_HORA_SESSAO'] == data)
    ]
    
    print(f"\n  {cpb[:10]}... | {sala} | {data}")
    print(f"    Existe no parquet? {'SIM' if len(match) > 0 else 'NÃO'}")
    
    if len(match) > 0:
        print(f"    ⚠️ EXISTE mas não deu match no join!")
        print(f"    Tipo no parquet: {match.iloc[0]['TIPO_SESSAO']}")
        print(f"    Tipos de dados:")
        print(f"      df_sessoes - CPB: {type(cpb)} | SALA: {type(sala)} | DATA: {type(data)}")
        match_row = match.iloc[0]
        print(f"      parquet    - CPB: {type(match_row['CPB_ROE'])} | SALA: {type(match_row['REGISTRO_SALA'])} | DATA: {type(match_row['DATA_HORA_SESSAO'])}")
        
        # Comparar valores exatos
        print(f"    Comparação de valores:")
        print(f"      CPB igual? {cpb == match_row['CPB_ROE']}")
        print(f"      SALA igual? {sala == match_row['REGISTRO_SALA']}")
        print(f"      DATA igual? {data == match_row['DATA_HORA_SESSAO']}")
        
        break  # Se achar um, parar para analisar

# 2. Verificar se há problema de tipo de dado
print("\n" + "="*70)
print("🔍 Teste 2: Comparar tipos de dados nas chaves")
print("="*70)

print(f"\ndf_sessoes:")
print(f"  CPB_ROE: {df_sessoes['CPB_ROE'].dtype}")
print(f"  REGISTRO_SALA: {df_sessoes['REGISTRO_SALA'].dtype}")
print(f"  DATA_HORA_SESSAO: {df_sessoes['DATA_HORA_SESSAO'].dtype}")

print(f"\ndf_tipos (no join):")
print(f"  CPB_ROE: {df_tipos['CPB_ROE'].dtype}")
print(f"  REGISTRO_SALA: {df_tipos['REGISTRO_SALA'].dtype}")
print(f"  DATA_HORA_SESSAO: {df_tipos['DATA_HORA_SESSAO'].dtype}")

print(f"\ndf_tipos_check (recém carregado):")
print(f"  CPB_ROE: {df_tipos_check['CPB_ROE'].dtype}")
print(f"  REGISTRO_SALA: {df_tipos_check['REGISTRO_SALA'].dtype}")
print(f"  DATA_HORA_SESSAO: {df_tipos_check['DATA_HORA_SESSAO'].dtype}")

INVESTIGAÇÃO DOS 32.618 FALTANTES

📊 Total de faltantes: 32,618

🔍 Teste 1: Verificar existência no parquet

Testando 10 sessões faltantes:

  E220043120... | 5002529 | 2023-01-05 16:00:00
    Existe no parquet? NÃO

  E220040360... | 5005624 | 2023-01-05 16:30:00
    Existe no parquet? NÃO

  E220040360... | 5005624 | 2023-01-05 19:00:00
    Existe no parquet? NÃO

  E220043120... | 5002529 | 2023-01-05 20:00:00
    Existe no parquet? NÃO

  E220041330... | 5005624 | 2023-01-05 21:30:00
    Existe no parquet? NÃO

  E220043120... | 5002529 | 2023-01-06 16:00:00
    Existe no parquet? NÃO

  E220040360... | 5005624 | 2023-01-06 16:30:00
    Existe no parquet? NÃO

  E220040360... | 5005624 | 2023-01-06 19:00:00
    Existe no parquet? NÃO

  E220043120... | 5002529 | 2023-01-06 20:00:00
    Existe no parquet? NÃO

  E220041330... | 5005624 | 2023-01-06 21:30:00
    Existe no parquet? NÃO

🔍 Teste 2: Comparar tipos de dados nas chaves

df_sessoes:
  CPB_ROE: object
  REGISTRO_SALA: objec

In [17]:
print("="*70)
print("REMOVENDO SESSÕES SEM TIPO_SESSAO")
print("="*70)

registros_antes = len(df_sessoes)
sem_tipo_antes = df_sessoes['TIPO_SESSAO'].isna().sum()

print(f"\n📊 Situação atual:")
print(f"  Total de sessões: {registros_antes:,}")
print(f"  Sem tipo: {sem_tipo_antes:,} ({sem_tipo_antes/registros_antes*100:.1f}%)")

# Remover sessões sem tipo
df_sessoes = df_sessoes[df_sessoes['TIPO_SESSAO'].notna()].copy()

registros_depois = len(df_sessoes)
removidos = registros_antes - registros_depois

print(f"\n✂️ Remoção concluída:")
print(f"  Sessões removidas: {removidos:,}")
print(f"  Sessões restantes: {registros_depois:,}")

print(f"\n✅ Agora temos 100% de cobertura de TIPO_SESSAO")
print(f"\n📊 Distribuição final:")
print(df_sessoes['TIPO_SESSAO'].value_counts())

print(f"\n📅 Período final:")
print(f"  {df_sessoes['DATA_HORA_SESSAO'].min()} até {df_sessoes['DATA_HORA_SESSAO'].max()}")

REMOVENDO SESSÕES SEM TIPO_SESSAO

📊 Situação atual:
  Total de sessões: 11,107,449
  Sem tipo: 32,618 (0.3%)

✂️ Remoção concluída:
  Sessões removidas: 32,618
  Sessões restantes: 11,074,831

✅ Agora temos 100% de cobertura de TIPO_SESSAO

📊 Distribuição final:
TIPO_SESSAO
Sessão Regular        10997867
Sessão Privada           37579
Pré-Estreia              36980
Mostra ou Festival        2405
Name: count, dtype: int64

📅 Período final:
  2023-01-05 04:20:00 até 2025-08-22 22:10:00


### 4.2. Mapear Tipo de Sessão → Modalidade

In [18]:
print("🔄 Mapeando tipos de sessão para modalidades...\n")

# Mapa de conversão
mapa_modalidade = {
    'Sessão Regular': 'A',
    'Pré-Estreia': 'B',
    'Mostra ou Festival': 'C',
    'Sessão Privada': 'D'
}

# Criar coluna MODALIDADE
df_sessoes['MODALIDADE'] = df_sessoes['TIPO_SESSAO'].map(mapa_modalidade)

# Para sessões sem tipo, assumir 'A' (Regular)
df_sessoes['MODALIDADE'].fillna('A', inplace=True)

print("✓ Modalidades criadas")
print(f"\n📊 Distribuição:")
modalidade_desc = {
    'A': 'Regular',
    'B': 'Pré-Estreia',
    'C': 'Mostra/Festival',
    'D': 'Privada'
}
for mod, count in df_sessoes['MODALIDADE'].value_counts().items():
    pct = count / len(df_sessoes) * 100
    desc = modalidade_desc.get(mod, 'Desconhecida')
    print(f"  {mod} ({desc}): {count:,} ({pct:.1f}%)")

🔄 Mapeando tipos de sessão para modalidades...

✓ Modalidades criadas

📊 Distribuição:
  A (Regular): 10,997,867 (99.3%)
  D (Privada): 37,579 (0.3%)
  B (Pré-Estreia): 36,980 (0.3%)
  C (Mostra/Festival): 2,405 (0.0%)


### 4.3. Extrair Nacionalidade do CPB/ROE

In [19]:
def extrair_nacionalidade_cpb(cpb):
    """
    Extrai nacionalidade da primeira letra do CPB/ROE.
    
    B = Brasileira
    E = Estrangeira
    G = Especial
    """
    if pd.isna(cpb):
        return None
    
    cpb_str = str(cpb).strip()
    if len(cpb_str) >= 1:
        primeira_letra = cpb_str[0].upper()
        if primeira_letra == 'B':
            return 'Brasileira'
        elif primeira_letra == 'E':
            return 'Estrangeira'
        elif primeira_letra == 'G':
            return 'Especial'
    return None

print("🏷️ Extraindo nacionalidade do CPB/ROE...")

df_sessoes['NACIONALIDADE'] = df_sessoes['CPB_ROE'].apply(extrair_nacionalidade_cpb)

print("\n✓ Nacionalidades extraídas")
print(f"\n📊 Distribuição:")
print(df_sessoes['NACIONALIDADE'].value_counts(dropna=False))

🏷️ Extraindo nacionalidade do CPB/ROE...

✓ Nacionalidades extraídas

📊 Distribuição:
NACIONALIDADE
Estrangeira    9675037
Brasileira     1344473
Especial         55321
Name: count, dtype: int64


### 4.4. Filtrar por Nacionalidade Válida

In [20]:
print("🔍 Filtrando por nacionalidades válidas...\n")

nacionalidades_validas = ['Brasileira', 'Estrangeira', 'Especial']

registros_antes = len(df_sessoes)
df_sessoes = df_sessoes[df_sessoes['NACIONALIDADE'].isin(nacionalidades_validas)].copy()
registros_removidos = registros_antes - len(df_sessoes)

print(f"✓ Sessões mantidas: {len(df_sessoes):,}")
print(f"✗ Sessões removidas (nacionalidade inválida): {registros_removidos:,}")

🔍 Filtrando por nacionalidades válidas...

✓ Sessões mantidas: 11,074,831
✗ Sessões removidas (nacionalidade inválida): 0


### 4.5. Join: Sessões + Salas

In [21]:
print("="*70)
print("JOIN: SESSÕES + SALAS")
print("="*70)

print(f"\nAntes do join:")
print(f"  Sessões: {len(df_sessoes):,} registros")
print(f"  Salas: {len(df_salas):,} registros")

# Join
df_sessoes = df_sessoes.merge(
    df_salas,
    on='REGISTRO_SALA',
    how='left',
    suffixes=('', '_sala')
)

print(f"\nApós o join:")
print(f"  Sessões: {len(df_sessoes):,} registros")

# Verificar cobertura
com_assentos = df_sessoes['ASSENTOS_SALA'].notna().sum()
sem_assentos = df_sessoes['ASSENTOS_SALA'].isna().sum()

print(f"\n📊 Cobertura de informação de salas:")
print(f"  Com ASSENTOS_SALA: {com_assentos:,} ({com_assentos/len(df_sessoes)*100:.1f}%)")
print(f"  Sem ASSENTOS_SALA: {sem_assentos:,} ({sem_assentos/len(df_sessoes)*100:.1f}%)")

JOIN: SESSÕES + SALAS

Antes do join:
  Sessões: 11,074,831 registros
  Salas: 6,303 registros

Após o join:
  Sessões: 11,074,831 registros

📊 Cobertura de informação de salas:
  Com ASSENTOS_SALA: 11,074,831 (100.0%)
  Sem ASSENTOS_SALA: 0 (0.0%)


### 4.6. Join: Sessões + IBGE (opcional)

In [22]:
print("="*70)
print("JOIN: SESSÕES + DADOS IBGE")
print("="*70)

# 1. Preparar df_ibge
print("\n1. Preparando df_ibge para mapeamento...")
df_ibge['municipio_limpo'] = df_ibge['nome_municipio'].str.replace(r'\s*\([A-Z]{2}\)\s*$', '', regex=True).str.upper().str.strip()
df_ibge['uf_extraida'] = df_ibge['nome_municipio'].str.extract(r'\(([A-Z]{2})\)$')[0]

# 2. Preparar df_sessoes
print("2. Preparando df_sessoes...")

# CORREÇÃO: Trocar hífen por espaço em MOGI-GUAÇU
df_sessoes.loc[df_sessoes['MUNICIPIO_SALA_COMPLEXO'] == 'MOGI-GUAÇU', 'MUNICIPIO_SALA_COMPLEXO'] = 'MOGI GUAÇU'

# Criar municipio_limpo
df_sessoes['municipio_limpo'] = df_sessoes['MUNICIPIO_SALA_COMPLEXO'].str.upper().str.strip()

# 3. Remover colunas IBGE antigas
print("3. Limpando colunas antigas...")
df_sessoes = df_sessoes.drop(columns=['codigo_municipio', 'nome_municipio', 'populacao', 'pib_total', 'pib_per_capita'], errors='ignore')

# 4. Join
print("4. Executando join por MUNICÍPIO + UF...")
df_sessoes = df_sessoes.merge(
    df_ibge[['municipio_limpo', 'uf_extraida', 'codigo_municipio', 'populacao', 'pib_total', 'pib_per_capita']],
    left_on=['municipio_limpo', 'UF_SALA_COMPLEXO'],
    right_on=['municipio_limpo', 'uf_extraida'],
    how='left'
)

# 5. Limpar colunas temporárias
df_sessoes = df_sessoes.drop(columns=['municipio_limpo', 'uf_extraida'], errors='ignore')

# 6. Verificar resultado
com_codigo = df_sessoes['codigo_municipio'].notna().sum()
com_pop = df_sessoes['populacao'].notna().sum()
com_pib = df_sessoes['pib_per_capita'].notna().sum()

print(f"\n✅ Join concluído!")
print(f"\n📊 Cobertura IBGE:")
print(f"  Com código IBGE: {com_codigo:,} ({com_codigo/len(df_sessoes)*100:.1f}%)")
print(f"  Com população: {com_pop:,} ({com_pop/len(df_sessoes)*100:.1f}%)")
print(f"  Com PIB per capita: {com_pib:,} ({com_pib/len(df_sessoes)*100:.1f}%)")

sem_match = df_sessoes[df_sessoes['codigo_municipio'].isna()]['MUNICIPIO_SALA_COMPLEXO'].unique()
if len(sem_match) > 0:
    print(f"\n⚠️ {len(sem_match)} municípios sem match: {sem_match}")
else:
    print(f"\n✅ 100% de cobertura alcançada!")

JOIN: SESSÕES + DADOS IBGE

1. Preparando df_ibge para mapeamento...
2. Preparando df_sessoes...
3. Limpando colunas antigas...
4. Executando join por MUNICÍPIO + UF...

✅ Join concluído!

📊 Cobertura IBGE:
  Com código IBGE: 11,061,998 (99.9%)
  Com população: 11,061,998 (99.9%)
  Com PIB per capita: 11,061,998 (99.9%)

⚠️ 1 municípios sem match: ['MOGI-GUACU']


## 5. Aplicação de Regras de Negócio

### 5.1. Criar Título Unificado

In [23]:
def criar_titulo_obra(row):
    """
    Prioriza TITULO_BRASIL, senão usa TITULO_ORIGINAL.
    """
    if pd.notna(row.get('TITULO_BRASIL')) and str(row.get('TITULO_BRASIL')).strip():
        return str(row['TITULO_BRASIL']).strip()
    return str(row.get('TITULO_ORIGINAL', '')).strip()

print("📝 Criando TITULO_OBRA unificado...")

df_sessoes['TITULO_OBRA'] = df_sessoes.apply(criar_titulo_obra, axis=1)

titulos_brasil = df_sessoes['TITULO_BRASIL'].notna().sum()
titulos_original = len(df_sessoes) - titulos_brasil

print(f"\n✓ Campo criado")
print(f"  Títulos do Brasil: {titulos_brasil:,} ({titulos_brasil/len(df_sessoes)*100:.1f}%)")
print(f"  Títulos originais: {titulos_original:,} ({titulos_original/len(df_sessoes)*100:.1f}%)")

📝 Criando TITULO_OBRA unificado...

✓ Campo criado
  Títulos do Brasil: 9,674,644 (87.4%)
  Títulos originais: 1,400,187 (12.6%)


### 5.2. Variáveis Temporais

In [24]:
print("📅 Criando variáveis temporais...\n")

# Extrair hora da sessão
df_sessoes['HORA_SESSAO'] = pd.to_datetime(df_sessoes['DATA_HORA_SESSAO']).dt.time

# Dia da semana
df_sessoes['DIA_SEMANA'] = df_sessoes['DATA_EXIBICAO'].dt.dayofweek

dias_semana = {
    0: 'Segunda', 1: 'Terça', 2: 'Quarta', 3: 'Quinta',
    4: 'Sexta', 5: 'Sábado', 6: 'Domingo'
}
df_sessoes['DIA_SEMANA_NOME'] = df_sessoes['DIA_SEMANA'].map(dias_semana)

# Fim de semana (quinta a domingo = 3, 4, 5, 6)
df_sessoes['FIM_DE_SEMANA'] = df_sessoes['DIA_SEMANA'].isin([3, 4, 5, 6])

print("✓ Variáveis criadas: HORA_SESSAO, DIA_SEMANA, DIA_SEMANA_NOME, FIM_DE_SEMANA")

📅 Criando variáveis temporais...

✓ Variáveis criadas: HORA_SESSAO, DIA_SEMANA, DIA_SEMANA_NOME, FIM_DE_SEMANA


In [25]:
def gerar_faixa_horaria(hora):
    """
    Classifica horário em faixas de 2 horas.
    """
    if pd.isna(hora):
        return 'Não informado'
    
    if isinstance(hora, pd.Timestamp):
        hora_int = hora.hour
    elif hasattr(hora, 'hour'):
        hora_int = hora.hour
    else:
        return 'Não informado'
    
    if 12 <= hora_int < 14:
        return '12-13h59'
    elif 14 <= hora_int < 16:
        return '14-15h59'
    elif 16 <= hora_int < 18:
        return '16-17h59'
    elif 18 <= hora_int < 20:
        return '18-19h59'
    elif 20 <= hora_int < 22:
        return '20-21h59'
    elif 22 <= hora_int < 24:
        return '22-23h59'
    else:
        return 'Outras'

print("⏰ Criando faixas horárias...")

df_sessoes['FAIXA_HORARIA'] = df_sessoes['DATA_HORA_SESSAO'].apply(gerar_faixa_horaria)

print("\n✓ Faixas criadas")
print(f"\n📊 Distribuição:")
print(df_sessoes['FAIXA_HORARIA'].value_counts())

⏰ Criando faixas horárias...

✓ Faixas criadas

📊 Distribuição:
FAIXA_HORARIA
20-21h59    2641152
18-19h59    2519957
16-17h59    2392295
14-15h59    2273065
12-13h59     863066
22-23h59     325548
Outras        59748
Name: count, dtype: int64


In [26]:
def calcular_semanas_cinema_vetorizado(df):
    """
    Versão vetorizada - processa todo o DataFrame de uma vez.
    Muito mais rápido que apply() linha por linha.
    """
    print("🚀 Calculando semanas cinematográficas (versão vetorizada)...")
    
    # Converter para datetime se necessário
    df['DATA_EXIBICAO'] = pd.to_datetime(df['DATA_EXIBICAO'])
    
    # Calcular dia da semana (0=Segunda, 6=Domingo)
    df['dia_semana'] = df['DATA_EXIBICAO'].dt.dayofweek
    
    # Calcular dias até a quinta-feira anterior
    # Se é quinta (3), sexta (4), sábado (5), domingo (6) ou segunda (0), terça (1), quarta (2)
    dias_ate_quinta = (df['dia_semana'] - 3) % 7
    
    # Data da quinta-feira que inicia a semana
    df['primeiro_dia_semana'] = df['DATA_EXIBICAO'] - pd.to_timedelta(dias_ate_quinta, unit='D')
    
    # Último dia (quarta-feira seguinte)
    df['ultimo_dia_semana'] = df['primeiro_dia_semana'] + pd.Timedelta(days=6)
    
    # Número da semana no ano
    df['semana_cinematografica'] = df['primeiro_dia_semana'].dt.isocalendar().week
    
    # Ano cinematográfico (ano da quinta-feira que inicia a semana)
    df['ano_cinematografico'] = df['primeiro_dia_semana'].dt.year
    
    # Limpar coluna temporária
    df.drop(columns=['dia_semana'], inplace=True)
    
    print("✓ Semanas calculadas!")
    return df

# Aplicar
df_sessoes = calcular_semanas_cinema_vetorizado(df_sessoes)

🚀 Calculando semanas cinematográficas (versão vetorizada)...
✓ Semanas calculadas!


In [27]:
print("="*70)
print("LIMPEZA DE MEMÓRIA")
print("="*70)

import gc

# Variáveis para deletar
vars_to_delete = [
    'df_tipos',
    'df_ibge',
    'df_salas',
    'mogi_sessoes',
    'mogi_ibge',
    'df_faltantes',
    'df_faltantes_inv',
    'df_tipos_check',
    'sem_tipo_df'
]

for var in vars_to_delete:
    if var in globals():
        del globals()[var]
        print(f"  ✓ Deletado: {var}")

gc.collect()

print(f"\n✅ Variáveis deletadas")
print(f"💾 Memória atual do df_sessoes: {df_sessoes.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

LIMPEZA DE MEMÓRIA
  ✓ Deletado: df_tipos
  ✓ Deletado: df_ibge
  ✓ Deletado: df_salas
  ✓ Deletado: df_faltantes
  ✓ Deletado: df_faltantes_inv
  ✓ Deletado: df_tipos_check

✅ Variáveis deletadas
💾 Memória atual do df_sessoes: 25.40 GB


### 5.3. Ocupação e Inconsistências

In [28]:
print("="*70)
print("CÁLCULO DE OCUPAÇÃO E IDENTIFICAÇÃO DE INCONSISTÊNCIAS")
print("="*70)

# Calcular taxa de ocupação
print("\n1. Calculando ocupação...")
df_sessoes['OCUPACAO_BRUTA'] = df_sessoes['PUBLICO'] / df_sessoes['ASSENTOS_SALA']

# Identificar sessões inconsistentes (ocupação > 110%)
print("2. Identificando sessões inconsistentes...")
df_sessoes['SESSAO_INCONSISTENTE'] = df_sessoes['OCUPACAO_BRUTA'] > 1.1

# Estatísticas
total_inconsistentes = df_sessoes['SESSAO_INCONSISTENTE'].sum()
pct_inconsistentes = total_inconsistentes / len(df_sessoes) * 100

print(f"\n✅ Ocupação calculada!")
print(f"\n📊 Resultados:")
print(f"   Total de sessões: {len(df_sessoes):,}")
print(f"   Sessões inconsistentes: {total_inconsistentes:,} ({pct_inconsistentes:.3f}%)")
print(f"   Ocupação média: {df_sessoes['OCUPACAO_BRUTA'].mean():.2%}")
print(f"   Ocupação máxima: {df_sessoes['OCUPACAO_BRUTA'].max():.2%}")

print(f"\n💡 Nota: Sessões com ocupação > 110% indicam provável erro")
print(f"   no cadastro da capacidade da sala.")

CÁLCULO DE OCUPAÇÃO E IDENTIFICAÇÃO DE INCONSISTÊNCIAS

1. Calculando ocupação...
2. Identificando sessões inconsistentes...

✅ Ocupação calculada!

📊 Resultados:
   Total de sessões: 11,074,831
   Sessões inconsistentes: 13,037 (0.118%)
   Ocupação média: 16.17%
   Ocupação máxima: 6500.00%

💡 Nota: Sessões com ocupação > 110% indicam provável erro
   no cadastro da capacidade da sala.


In [29]:
print("="*70)
print("ANÁLISE: SALAS COM SESSÕES INCONSISTENTES")
print("="*70)

# Identificar salas que têm sessões inconsistentes
salas_inconsistentes = df_sessoes[df_sessoes['SESSAO_INCONSISTENTE'] == True].groupby('REGISTRO_SALA').agg({
    'SESSAO_INCONSISTENTE': 'count',  # Total de sessões inconsistentes
    'ASSENTOS_SALA': 'first',
    'NOME_SALA': 'first',
    'MUNICIPIO_COMPLEXO': 'first',
    'OCUPACAO_BRUTA': 'max',  # Maior ocupação
    'PUBLICO': 'max'  # Maior público
}).rename(columns={'SESSAO_INCONSISTENTE': 'sessoes_inconsistentes'})

salas_inconsistentes = salas_inconsistentes.sort_values('sessoes_inconsistentes', ascending=False)

print(f"\n📊 Total de salas com inconsistências: {len(salas_inconsistentes)}")
print(f"📊 Total de sessões inconsistentes: {df_sessoes['SESSAO_INCONSISTENTE'].sum():,}")

print("\n📋 Top 20 salas com mais sessões inconsistentes:")
print(salas_inconsistentes.head(20).to_string())

# Verificar se vale a pena corrigir
print("\n" + "="*70)
print("DECISÃO: VALE A PENA CORRIGIR?")
print("="*70)

total_inconsistentes = df_sessoes['SESSAO_INCONSISTENTE'].sum()
pct_inconsistentes = total_inconsistentes / len(df_sessoes) * 100

print(f"\n📊 Impacto:")
print(f"   Sessões inconsistentes: {total_inconsistentes:,} ({pct_inconsistentes:.3f}%)")
print(f"   Sessões OK: {len(df_sessoes) - total_inconsistentes:,} ({100-pct_inconsistentes:.3f}%)")

print(f"\n💡 Recomendação:")
if pct_inconsistentes < 0.5:
    print(f"   ✓ Impacto baixíssimo ({pct_inconsistentes:.3f}%)")
    print(f"   ✓ Pode MANTER a flag e seguir em frente")
    print(f"   ✓ Ou REMOVER essas {total_inconsistentes:,} sessões se preferir dados 100% confiáveis")
else:
    print(f"   ⚠️ Impacto significativo ({pct_inconsistentes:.2f}%)")
    print(f"   ⚠️ Vale a pena investigar e corrigir")

ANÁLISE: SALAS COM SESSÕES INCONSISTENTES

📊 Total de salas com inconsistências: 125
📊 Total de sessões inconsistentes: 13,037

📋 Top 20 salas com mais sessões inconsistentes:
               sessoes_inconsistentes  ASSENTOS_SALA                      NOME_SALA   MUNICIPIO_COMPLEXO  OCUPACAO_BRUTA  PUBLICO
REGISTRO_SALA                                                                                                                    
5006147                          3102              3                IGUATEMI FLN 01        FLORIANÓPOLIS            65.0      195
5002347                           579             52           CINE CIDADE JARDIM 4            SÃO PAULO        1.538462       80
5005067                           533             54  CINÉPOLIS PARQUE MAIA SALA 08            GUARULHOS        6.666667      360
5003036                           466             51              SALA ALPHA 03 VIP              BARUERI        1.372549       70
5003386                           462       

In [30]:
print("="*70)
print("GERANDO RELATÓRIO DE SALAS COM INCONSISTÊNCIAS")
print("="*70)

# Preparar dados do relatório
salas_problema = df_sessoes[df_sessoes['SESSAO_INCONSISTENTE'] == True].groupby('REGISTRO_SALA').agg({
    'NOME_SALA': 'first',
    'MUNICIPIO_COMPLEXO': 'first',
    'UF_COMPLEXO': 'first',
    'ASSENTOS_SALA': 'first',
    'SESSAO_INCONSISTENTE': 'count',  # Total de sessões inconsistentes
    'PUBLICO': ['max', 'mean'],
    'OCUPACAO_BRUTA': 'max'
}).reset_index()

# Renomear colunas
salas_problema.columns = [
    'REGISTRO_SALA',
    'NOME_SALA',
    'MUNICIPIO',
    'UF',
    'ASSENTOS_CADASTRADOS',
    'SESSOES_INCONSISTENTES',
    'PUBLICO_MAXIMO',
    'PUBLICO_MEDIO',
    'OCUPACAO_MAXIMA'
]

# Sugerir capacidade corrigida (percentil 95)
capacidade_sugerida = []
for registro in salas_problema['REGISTRO_SALA']:
    sessoes = df_sessoes[df_sessoes['REGISTRO_SALA'] == registro]
    cap_sugerida = int(sessoes['PUBLICO'].quantile(0.95))
    capacidade_sugerida.append(cap_sugerida)

salas_problema['CAPACIDADE_SUGERIDA'] = capacidade_sugerida

# Ordenar por número de sessões inconsistentes
salas_problema = salas_problema.sort_values('SESSOES_INCONSISTENTES', ascending=False)

# Formatar ocupação como percentual
salas_problema['OCUPACAO_MAXIMA'] = (salas_problema['OCUPACAO_MAXIMA'] * 100).round(1)

# Salvar
caminho_relatorio = '../Bases/relatorio_salas_inconsistentes.xlsx'
salas_problema.to_excel(caminho_relatorio, index=False, sheet_name='Salas Inconsistentes')

print(f"\n✅ Relatório gerado: {caminho_relatorio}")
print(f"   📊 Total de salas: {len(salas_problema)}")
print(f"   📊 Total de sessões afetadas: {salas_problema['SESSOES_INCONSISTENTES'].sum():,}")
print(f"\n📋 Preview:")
print(salas_problema.head(10))

GERANDO RELATÓRIO DE SALAS COM INCONSISTÊNCIAS

✅ Relatório gerado: ../Bases/relatorio_salas_inconsistentes.xlsx
   📊 Total de salas: 125
   📊 Total de sessões afetadas: 13,037

📋 Preview:
    REGISTRO_SALA                      NOME_SALA            MUNICIPIO  UF  \
103       5006147                IGUATEMI FLN 01        FLORIANÓPOLIS  SC   
47        5002347           CINE CIDADE JARDIM 4            SÃO PAULO  SP   
89        5005067  CINÉPOLIS PARQUE MAIA SALA 08            GUARULHOS  SP   
61        5003036              SALA ALPHA 03 VIP              BARUERI  SP   
65        5003386    MULTIPLEX MINAS SHOPPING 01       BELO HORIZONTE  MG   
25        5000922      MOVIECOM PRUDENSHOPPING 1  PRESIDENTE PRUDENTE  SP   
79        5004464      SALA NATAL SHOPPING 5 VIP                NATAL  RN   
31        5001488      MOVIECOM PRUDENSHOPPING 2  PRESIDENTE PRUDENTE  SP   
53        5002587                CINE PARAÍSO 05            SÃO PAULO  SP   
70        5003492       SALA CINEPOLIS JK

In [31]:
print("="*70)
print("COLUNAS DISPONÍVEIS EM df_sessoes")
print("="*70)

print(f"\n📋 Total de colunas: {len(df_sessoes.columns)}")
print("\nLista completa:")
for i, col in enumerate(df_sessoes.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n" + "="*70)
print("PRIMEIRAS 3 LINHAS")
print("="*70)
print(df_sessoes.head(3).to_string())

COLUNAS DISPONÍVEIS EM df_sessoes

📋 Total de colunas: 56

Lista completa:
   1. DATA_EXIBICAO
   2. DATA_HORA_SESSAO
   3. TITULO_ORIGINAL
   4. TITULO_BRASIL
   5. CPB_ROE
   6. AUDIO
   7. LEGENDADA
   8. PAIS_OBRA
   9. REGISTRO_SALA
  10. PUBLICO
  11. REGISTRO_GRUPO_EXIBIDOR
  12. REGISTRO_EXIBIDOR
  13. REGISTRO_COMPLEXO
  14. MUNICIPIO_SALA_COMPLEXO
  15. UF_SALA_COMPLEXO
  16. RAZAO_SOCIAL_EXIBIDORA
  17. ANO_CINEMA
  18. TIPO_SESSAO
  19. chave_completa
  20. MODALIDADE
  21. NACIONALIDADE
  22. NOME_SALA
  23. CNPJ_SALA
  24. SITUACAO_SALA
  25. DATA_SITUACAO_SALA
  26. DATA_INICIO_FUNCIONAMENTO_SALA
  27. ASSENTOS_SALA
  28. NOME_COMPLEXO
  29. REGISTRO_COMPLEXO_sala
  30. SITUACAO_COMPLEXO
  31. DATA_SITUACAO_COMPLEXO
  32. MUNICIPIO_COMPLEXO
  33. UF_COMPLEXO
  34. COMPLEXO_ITINERANTE
  35. OPERACAO_USUAL
  36. NOME_EXIBIDOR
  37. REGISTRO_EXIBIDOR_sala
  38. CNPJ_EXIBIDOR
  39. SITUACAO_EXIBIDOR
  40. NOME_GRUPO_EXIBIDOR
  41. codigo_municipio
  42. populacao
  43. pib_t

In [32]:
print("="*70)
print("ANÁLISE: COLUNAS DUPLICADAS")
print("="*70)

# Identificar colunas duplicadas (_x, _y, e sem sufixo)
colunas_suspeitas = [
    'populacao', 'populacao_x', 'populacao_y',
    'pib_total', 'pib_total_x', 'pib_total_y',
    'pib_per_capita', 'pib_per_capita_x', 'pib_per_capita_y'
]

print("\n🔍 Verificando quais existem e seus valores:")
for col in colunas_suspeitas:
    if col in df_sessoes.columns:
        nulos = df_sessoes[col].isna().sum()
        nao_nulos = len(df_sessoes) - nulos
        print(f"\n  {col}:")
        print(f"    Existe: SIM")
        print(f"    Não-nulos: {nao_nulos:,} ({nao_nulos/len(df_sessoes)*100:.1f}%)")
        print(f"    Valores únicos: {df_sessoes[col].nunique()}")
        if nao_nulos > 0:
            print(f"    Exemplo: {df_sessoes[col].dropna().iloc[0]}")
    else:
        print(f"\n  {col}: NÃO EXISTE")

# Comparar valores entre as versões
print("\n" + "="*70)
print("COMPARAÇÃO: AS COLUNAS TÊM OS MESMOS VALORES?")
print("="*70)

if 'populacao' in df_sessoes.columns and 'populacao_x' in df_sessoes.columns:
    diferentes = (df_sessoes['populacao'] != df_sessoes['populacao_x']).sum()
    print(f"\n  populacao vs populacao_x:")
    print(f"    Valores diferentes: {diferentes:,}")
    
if 'populacao' in df_sessoes.columns and 'populacao_y' in df_sessoes.columns:
    diferentes = (df_sessoes['populacao'] != df_sessoes['populacao_y']).sum()
    print(f"\n  populacao vs populacao_y:")
    print(f"    Valores diferentes: {diferentes:,}")

print("\n💡 Decisão: SÓ deletar se:")
print("   1. As colunas _x e _y têm exatamente os mesmos valores que a sem sufixo")
print("   2. OU se as _x e _y estão vazias/inúteis")

ANÁLISE: COLUNAS DUPLICADAS

🔍 Verificando quais existem e seus valores:

  populacao:
    Existe: SIM
    Não-nulos: 11,061,998 (99.9%)
    Valores únicos: 457
    Exemplo: 81506.0

  populacao_x: NÃO EXISTE

  populacao_y: NÃO EXISTE

  pib_total:
    Existe: SIM
    Não-nulos: 11,061,998 (99.9%)
    Valores únicos: 458
    Exemplo: 1732690.0

  pib_total_x: NÃO EXISTE

  pib_total_y: NÃO EXISTE

  pib_per_capita:
    Existe: SIM
    Não-nulos: 11,061,998 (99.9%)
    Valores únicos: 458
    Exemplo: 21258.4349618433

  pib_per_capita_x: NÃO EXISTE

  pib_per_capita_y: NÃO EXISTE

COMPARAÇÃO: AS COLUNAS TÊM OS MESMOS VALORES?

💡 Decisão: SÓ deletar se:
   1. As colunas _x e _y têm exatamente os mesmos valores que a sem sufixo
   2. OU se as _x e _y estão vazias/inúteis


In [33]:
print("="*70)
print("INVESTIGAÇÃO: 12.833 REGISTROS COM VALORES DIFERENTES")
print("="*70)

# Identificar registros onde populacao != populacao_y
diferentes = df_sessoes[df_sessoes['populacao'] != df_sessoes['populacao_y']].copy()

print(f"\n📊 Total de registros com diferença: {len(diferentes):,}")

# Ver exemplos
print("\n📋 Exemplos:")
print(diferentes[['MUNICIPIO_SALA_COMPLEXO', 'populacao', 'populacao_y', 'codigo_municipio']].head(10))

# Ver quais municípios têm diferença
municipios_diferentes = diferentes['MUNICIPIO_SALA_COMPLEXO'].unique()
print(f"\n🗺️ Municípios com diferença: {len(municipios_diferentes)}")
print(f"   {municipios_diferentes}")

# Verificar se é o caso de MOGI-GUAÇU
print("\n❓ Hipótese: São as 12.833 sessões de MOGI-GUAÇU que corrigimos?")
mogi = df_sessoes[df_sessoes['MUNICIPIO_SALA_COMPLEXO'] == 'MOGI GUAÇU']
print(f"   Total de sessões em MOGI GUAÇU: {len(mogi):,}")

INVESTIGAÇÃO: 12.833 REGISTROS COM VALORES DIFERENTES


KeyError: 'populacao_y'

In [ ]:
print("="*70)
print("LIMPEZA: REMOVENDO COLUNAS _x E _y")
print("="*70)

colunas_antes = len(df_sessoes.columns)

# Remover todas as colunas com sufixo _x ou _y
colunas_remover = [col for col in df_sessoes.columns if col.endswith('_x') or col.endswith('_y')]

print(f"\n📋 Colunas a remover ({len(colunas_remover)}):")
for col in colunas_remover:
    print(f"   • {col}")

df_sessoes = df_sessoes.drop(columns=colunas_remover)

print(f"\n✅ Limpeza concluída!")
print(f"   Colunas: {colunas_antes} → {len(df_sessoes.columns)}")
print(f"   Removidas: {len(colunas_remover)}")

# Verificar colunas finais de IBGE
print(f"\n📊 Colunas IBGE finais:")
colunas_ibge = [col for col in df_sessoes.columns if any(x in col.lower() for x in ['populacao', 'pib', 'codigo_municipio'])]
for col in colunas_ibge:
    nulos = df_sessoes[col].isna().sum()
    print(f"   • {col}: {len(df_sessoes) - nulos:,} não-nulos ({(len(df_sessoes) - nulos)/len(df_sessoes)*100:.1f}%)")

## 6. Validações

### 6.1. Campos Obrigatórios

In [ ]:
print("="*70)
print("LIMPEZA: REMOVENDO COLUNAS DUPLICADAS (_x e _y)")
print("="*70)

print("\n💡 Contexto:")
print("   Durante os joins com IBGE, foram criadas colunas duplicadas:")
print("   • _x: completamente vazias (lixo)")
print("   • _y: desatualizadas (antes de corrigir MOGI-GUAÇU)")
print("   • sem sufixo: corretas e completas (100% cobertura)")

colunas_antes = len(df_sessoes.columns)

# Remover todas as colunas com sufixo _x ou _y
colunas_remover = [col for col in df_sessoes.columns if col.endswith('_x') or col.endswith('_y')]

if len(colunas_remover) > 0:
    print(f"\n📋 Colunas a remover ({len(colunas_remover)}):")
    for col in colunas_remover:
        print(f"   • {col}")
    
    df_sessoes = df_sessoes.drop(columns=colunas_remover)
    
    print(f"\n✅ Limpeza concluída!")
    print(f"   Colunas: {colunas_antes} → {len(df_sessoes.columns)}")
    print(f"   Removidas: {len(colunas_remover)}")
else:
    print("\n✓ Não há colunas duplicadas para remover")

# Verificar colunas finais de IBGE
print(f"\n📊 Validação: Colunas IBGE finais")
colunas_ibge = [col for col in df_sessoes.columns if any(x in col.lower() for x in ['populacao', 'pib', 'codigo_municipio'])]
for col in colunas_ibge:
    nulos = df_sessoes[col].isna().sum()
    print(f"   • {col}: {len(df_sessoes) - nulos:,} não-nulos ({(len(df_sessoes) - nulos)/len(df_sessoes)*100:.1f}%)")

### 6.2. Estatísticas Finais

In [ ]:
print("\n" + "="*70)
print("ESTATÍSTICAS FINAIS DO DATASET")
print("="*70)

print(f"\n📊 Dimensões:")
print(f"  Total de sessões: {len(df_sessoes):,}")
print(f"  Total de colunas: {len(df_sessoes.columns)}")

print(f"\n🎬 Obras:")
print(f"  Obras únicas: {df_sessoes['TITULO_OBRA'].nunique():,}")
print(f"  CPBs únicos: {df_sessoes['CPB_ROE'].nunique():,}")

print(f"\n🏢 Infraestrutura:")
print(f"  Salas únicas: {df_sessoes['REGISTRO_SALA'].nunique():,}")
print(f"  Complexos únicos: {df_sessoes['REGISTRO_COMPLEXO'].nunique():,}")
print(f"  Exibidores únicos: {df_sessoes['REGISTRO_EXIBIDOR'].nunique():,}")

print(f"\n👥 Público:")
print(f"  Público total: {df_sessoes['PUBLICO'].sum():,}")
print(f"  Público médio/sessão: {df_sessoes['PUBLICO'].mean():.1f}")
print(f"  Sessões com público zero: {(df_sessoes['PUBLICO'] == 0).sum():,}")

print(f"\n🌍 Distribuição por nacionalidade:")
for nac, count in df_sessoes['NACIONALIDADE'].value_counts().items():
    pct = count / len(df_sessoes) * 100
    print(f"  {nac}: {count:,} ({pct:.1f}%)")

print(f"\n🎫 Distribuição por modalidade:")
for mod, count in df_sessoes['MODALIDADE'].value_counts().items():
    pct = count / len(df_sessoes) * 100
    desc = modalidade_desc.get(mod, 'Desconhecida')
    print(f"  {mod} ({desc}): {count:,} ({pct:.1f}%)")

print(f"\n📅 Por ano cinematográfico:")
for ano, count in df_sessoes['ANO_CINEMA'].value_counts().sort_index().items():
    pct = count / len(df_sessoes) * 100
    print(f"  {ano}: {count:,} ({pct:.1f}%)")

## 7. Preparação Final e Exportação

### 7.1. Selecionar e Ordenar Colunas

In [ ]:
print("="*70)
print("ORGANIZAÇÃO FINAL DAS COLUNAS")
print("="*70)

# Definir ordem lógica das colunas
colunas_finais = [
    # Identificadores temporais
    'DATA_EXIBICAO',
    'DATA_HORA_SESSAO',
    'ANO_CINEMA',
    'ano_cinematografico',
    'semana_cinematografica',
    'primeiro_dia_semana',
    'ultimo_dia_semana',
    'HORA_SESSAO',
    'DIA_SEMANA',
    'DIA_SEMANA_NOME',
    'FIM_DE_SEMANA',
    'FAIXA_HORARIA',
    
    # Obra
    'CPB_ROE',
    'TITULO_OBRA',
    'TITULO_ORIGINAL',
    'TITULO_BRASIL',
    'NACIONALIDADE',
    'PAIS_OBRA',
    
    # Sessão
    'TIPO_SESSAO',
    'MODALIDADE',
    'AUDIO',
    'LEGENDADA',
    
    # Sala - Identificação
    'REGISTRO_SALA',
    'NOME_SALA',
    'CNPJ_SALA',
    'SITUACAO_SALA',
    'DATA_SITUACAO_SALA',
    'DATA_INICIO_FUNCIONAMENTO_SALA',
    
    # Sala - Características físicas
    'ASSENTOS_SALA',
    
    # Complexo - Identificação
    'REGISTRO_COMPLEXO',
    'REGISTRO_COMPLEXO_sala',
    'NOME_COMPLEXO',
    'MUNICIPIO_SALA_COMPLEXO',
    'UF_SALA_COMPLEXO',
    'MUNICIPIO_COMPLEXO',
    'UF_COMPLEXO',
    
    # Complexo - Características
    'SITUACAO_COMPLEXO',
    'DATA_SITUACAO_COMPLEXO',
    'COMPLEXO_ITINERANTE',
    'OPERACAO_USUAL',  # ← IMPORTANTE: Shopping, Centro, etc
    
    # Exibidor - Identificação
    'REGISTRO_EXIBIDOR',
    'REGISTRO_EXIBIDOR_sala',
    'NOME_EXIBIDOR',
    'CNPJ_EXIBIDOR',
    'RAZAO_SOCIAL_EXIBIDORA',
    'SITUACAO_EXIBIDOR',
    
    # Grupo Exibidor
    'REGISTRO_GRUPO_EXIBIDOR',
    'NOME_GRUPO_EXIBIDOR',
    
    # Métricas de Performance
    'PUBLICO',
    'OCUPACAO_BRUTA',
    'SESSAO_INCONSISTENTE',
    
    # Dados Socioeconômicos (IBGE)
    'codigo_municipio',
    'populacao',
    'pib_total',
    'pib_per_capita'
]

# Verificar quais existem
colunas_existentes = [col for col in colunas_finais if col in df_sessoes.columns]
colunas_faltantes = [col for col in colunas_finais if col not in df_sessoes.columns]

print(f"\n📋 Status das colunas:")
print(f"   ✓ Encontradas: {len(colunas_existentes)}")
if colunas_faltantes:
    print(f"   ⚠️ Não encontradas: {len(colunas_faltantes)}")
    for col in colunas_faltantes:
        print(f"      • {col}")

# Reorganizar
df_sessoes = df_sessoes[colunas_existentes]

print(f"\n✅ Dataset final organizado: {len(df_sessoes.columns)} colunas")

# Mostrar resumo por categoria
print(f"\n📊 Resumo por categoria:")
print(f"   • Temporais: 12 colunas")
print(f"   • Obra: 7 colunas")
print(f"   • Sessão: 4 colunas")
print(f"   • Sala: 7 colunas")
print(f"   • Complexo: 10 colunas (inclui OPERACAO_USUAL)")
print(f"   • Exibidor: 7 colunas")
print(f"   • Métricas: 3 colunas")
print(f"   • Socioeconômicos: 4 colunas")

### 7.2. Exportar Parquet

In [ ]:
print("="*70)
print("EXPORTAÇÃO")
print("="*70)

print(f"\n💾 Salvando {CAMINHO_OUTPUT}...")

# Criar diretório se não existir
Path(CAMINHO_OUTPUT).parent.mkdir(parents=True, exist_ok=True)

# Salvar
df_sessoes_limpo.to_parquet(CAMINHO_OUTPUT, index=False, compression='snappy')

tamanho = Path(CAMINHO_OUTPUT).stat().st_size / (1024 * 1024)
print(f"✓ Arquivo salvo ({tamanho:.2f} MB)")

### 7.3. Criar Dicionário de Dados

In [ ]:
print(f"\n📖 Criando dicionário de dados...")

dicionario = {
    "metadata": {
        "data_criacao": datetime.now().isoformat(),
        "anos_cinema": ANOS,
        "periodos": {str(k): v for k, v in PERIODOS.items()},
        "total_registros": len(df_sessoes_limpo),
        "total_colunas": len(df_sessoes_limpo.columns)
    },
    "colunas": {}
}

# Adicionar info de cada coluna
for col in df_sessoes_limpo.columns:
    dicionario['colunas'][col] = {
        'tipo': str(df_sessoes_limpo[col].dtype),
        'nulos': int(df_sessoes_limpo[col].isna().sum()),
        'pct_nulos': float(df_sessoes_limpo[col].isna().sum() / len(df_sessoes_limpo) * 100)
    }

# Salvar
with open(CAMINHO_DICIONARIO, 'w', encoding='utf-8') as f:
    json.dump(dicionario, f, indent=2, ensure_ascii=False)

print(f"✓ Dicionário salvo: {CAMINHO_DICIONARIO}")

## ✅ Conclusão

**Processamento concluído!**

Arquivos gerados:
- `df_sessoes_limpo.parquet` - Dataset principal
- `dicionario_df_sessoes_limpo.json` - Metadados

**Próximos passos:**
1. Carregar o parquet nos notebooks seguintes
2. Realizar agregações e análises
3. Usar ANO_CINEMA=2025 para teste/validação

In [ ]:
print("\n" + "="*70)
print("✅ PROCESSAMENTO CONCLUÍDO COM SUCESSO!")
print("="*70)
print(f"\n📊 Resumo final:")
print(f"   • Sessões processadas: {len(df_sessoes_limpo):,}")
print(f"   • Anos: {', '.join(map(str, ANOS))}")
print(f"   • Colunas: {len(df_sessoes_limpo.columns)}")
print(f"   • Arquivo: {CAMINHO_OUTPUT}")
print(f"\n🎬 Pronto para análises!")